In [1]:
import geopandas as gpd
import pandas as pd 
import numpy as np
import geemap, ee

import rasterio 
import matplotlib.pyplot as plt
import json

from tqdm import tqdm 
import os

import time

In [2]:
try:
        ee.Initialize()
        
except Exception as e:
    
        ee.Authenticate()
        ee.Initialize()

In [3]:
# read_file no acepta crs= (se ignora: ver warning). Se usa el CRS del .prj y se reproyecta a WGS84

blocks_peru = gpd.read_file(r"..\data\temporal_data\shapefile\DatasetGrids2km.shp").to_crs("EPSG:4326")

In [4]:
# -----------------------------------------------------------------------------
# Sentinel-2: RGB bands and cloud mask
# -----------------------------------------------------------------------------
# Collection: COPERNICUS/S2_HARMONIZED (Level-1C, top-of-atmosphere reflectance).
#   The atmospherically corrected product (S2_SR_HARMONIZED, Level-2A) barely exists for
#   Peru in 2018 (e.g. Lima: 7 images in the whole year, none Jun-Oct), while Level-1C has
#   ~6 images per month in every city. "HARMONIZED" = same scale before/after 2022.
# Bands (fixed order in every .tif = channel i of the CNN): 10 m resolution
#   B4 = Red, B3 = Green, B2 = Blue
# Values: reflectance x 10000 (integer, uint16). Divide by 10000 to get reflectance 0-1.
BANDS      = ["B4", "B3", "B2"]
BAND_NAMES = ["Red", "Green", "Blue"]

# Cloud mask: s2cloudless (COPERNICUS/S2_CLOUD_PROBABILITY), a cloud probability per pixel
# (0-100) computed by ESA/Sentinel Hub. Much better than the QA60 band of Level-1C, which is
# coarse (60 m) and misses thin clouds. Pixels with probability >= 40% are removed.
CLOUD_PROB_MAX = 40

def add_cloud_mask(image):
    prob = ee.Image(image.get("cloud_prob")).select("probability")
    return image.updateMask(prob.lt(CLOUD_PROB_MAX))

def s2_collection(aoi, start, end):
    """Sentinel-2 L1C images over `aoi`, each one joined to its cloud probability."""
    s2 = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
          .filterBounds(aoi).filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80)))      # drop almost fully cloudy scenes
    clouds = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
              .filterBounds(aoi).filterDate(start, end))
    joined = ee.Join.saveFirst("cloud_prob").apply(
        primary=s2, secondary=clouds,
        condition=ee.Filter.equals(leftField="system:index", rightField="system:index"))
    return ee.ImageCollection(joined).map(add_cloud_mask)

In [5]:
# blocks_peru = blocks_peru.iloc[4467:,]


In [6]:
# blocks_peru

In [ ]:
# -----------------------------------------------------------------------------
# Download one Sentinel-2 RGB image (10 m) per 2 km grid
# -----------------------------------------------------------------------------
# Period: ALL of 2018. Fixed months do not work for all of Peru: Lima has no clear images
# from June to October (winter fog) and Iquitos is clear mainly in July-September. With the
# whole year + cloud mask, each pixel takes the MEDIAN of its clear observations.
# Output: Sentinel2_grid2km/Grid_{ID}.tif, 3 bands (Red, Green, Blue), uint16, ~200 x 200
# pixels per 2 km grid. Pixels outside the grid polygon or without clear observations = 0
# (declared as nodata).
# 503 errors (Earth Engine busy) are retried up to 4 times, waiting longer each time.
import requests

START_DATE, END_DATE = "2018-01-01", "2019-01-01"     # END_DATE is exclusive -> all 2018
SCALE    = 10                                          # meters per pixel (Sentinel-2 RGB)
NODATA   = 0
RETRIES  = 4

out_dir = os.path.join("..", "data", "raw_data", "Sentinel2_grid2km")
os.makedirs(out_dir, exist_ok=True)

def download(url, path):
    """GET with retries for temporary server errors (429 / 5xx)."""
    for attempt in range(RETRIES):
        r = requests.get(url, timeout=300)
        if r.status_code in (429, 500, 502, 503, 504) and attempt < RETRIES - 1:
            time.sleep(5 * 2 ** attempt)                   # 5, 10, 20 s
            continue
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        return

failed = []
for index, row in tqdm(blocks_peru.iterrows(), total=len(blocks_peru)):

    out_path = os.path.join(out_dir, f"Grid_{row['ID']}.tif")

    # Allows resuming the download without repeating grids already saved
    if os.path.exists(out_path):
        continue

    # Square polygon of 4 km2 (EPSG:4326)
    aoi = ee.Geometry(row.geometry.__geo_interface__)
    col = s2_collection(aoi, START_DATE, END_DATE)

    try:
        if col.size().getInfo() == 0:
            failed.append((row["ID"], "no images"))
            continue

        # Daytime image: median of the clear observations, RGB, clipped to the polygon.
        # unmask AFTER clip (sameFootprint=False): pixels outside the polygon or without
        # clear observations become 0 instead of undefined values.
        img = (col.select(BANDS).median()
               .clip(aoi)
               .unmask(NODATA, False)
               .toUint16())

        url = img.getDownloadURL({
            "region": aoi,
            "scale": SCALE,
            "format": "GEO_TIFF",          # one multi-band .tif (no .zip)
        })
        download(url, out_path)

        # Band names and nodata written in the GeoTIFF
        with rasterio.open(out_path, "r+") as dst:
            dst.descriptions = tuple(BAND_NAMES)
            dst.nodata = NODATA

    except Exception as e:
        failed.append((row["ID"], str(e)[:120]))
        print(f"Error exporting image for grid {row['ID']}: {e}")
        continue

print(f"Grids that failed: {len(failed)} (re-run the cell to retry them)")

  1%|          | 72/6176 [08:16<15:10:03,  8.95s/it]